# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a walkthrough for loading and exploring the FAIR^2 dataset using the `mlcroissant` library and pandas. All dataset elements are referenced using their schema `@id` fields for clarity and reproducibility.

### Dataset Source
The dataset is published as a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant pandas

## 1. Data Loading

Load the dataset metadata and records using the `mlcroissant` library. The metadata provides essential information such as the dataset's name, description, and available record sets for further exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load dataset using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset Name: {metadata.name}\nDescription: {metadata.description}\nVersion: {getattr(metadata, 'version', '<no version>')}")

## 2. Data Overview

Review the available record sets (tables), their `@id` identifiers, and the field `@id`s for each record set. This ensures further data manipulation is referenced by their canonical identifiers.

We list all record sets and their associated fields below.

In [ ]:
# List all record sets by @id and their fields by @id
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"  • Record set name: {rs.name}")
    print(f"    @id: {rs.id}")
    print("    Fields:")
    for field in rs.fields:
        print(f"      - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', '<unknown>')})")
    print()

# Preview first record for each record set (referenced by @id)
for rs in record_sets:
    print(f"First record in record set '{rs.name}' (@id: {rs.id}):")
    try:
        record_iterator = dataset.records(record_set=rs.id)
        record = next(record_iterator)
        print(record)
    except StopIteration:
        print("  (No records found)")
    except Exception as e:
        print(f"  (Error retrieving records: {e})")
    print()

## 3. Data Extraction

For downstream analysis, load the main record set(s) into pandas DataFrames. Use the full `@id` to guarantee precise mapping. Replace or add more record sets below as needed.

In [ ]:
# Find main record set(s) based on field content (modify as appropriate for your dataset)

main_record_set_ids = [rs.id for rs in dataset.record_sets]
print(f"Loading all record sets: {main_record_set_ids}")
dataframes = {}
for record_set_id in main_record_set_ids:
    print(f"\nLoading records from record set: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded DataFrame with columns: {df.columns.tolist()}")
        print(df.head(3))
    else:
        print("  No records found.")

## 4. Exploratory Data Analysis (EDA)

We now perform some typical EDA and processing. This includes filtering records, normalizing fields, and grouping data by key attributes (using columns referred by their `@id`).

Please update the field `@id`s as needed for your workflow.

In [ ]:
# Select a record set (by @id) for EDA
record_set_id = next((rs.id for rs in dataset.record_sets if rs.fields), None)
df = dataframes.get(record_set_id)

# List all columns with their corresponding @id from Croissant
print(f"Available fields in record set {record_set_id}:")
for rs in dataset.record_sets:
    if rs.id == record_set_id:
        for field in rs.fields:
            print(f"  • Name: {field.name} -- @id: {field.id}, data type: {getattr(field, 'data_type', '<unknown>')}")
        break

# Example: Select a numeric field and a group field (update as needed)
# Let's automatically pick an integer or float field, and a categorical field
numeric_field_id = None
group_field_id = None
for rs in dataset.record_sets:
    if rs.id == record_set_id:
        for field in rs.fields:
            if getattr(field, 'data_type', '').lower() in ('integer', 'float', 'number'):
                numeric_field_id = field.id
            elif group_field_id is None and getattr(field, 'data_type', '').lower() in ('string', 'text'):
                group_field_id = field.id
        break

print(f"\n[INFO] Using numeric_field_id={numeric_field_id} and group_field_id={group_field_id}")

if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].quantile(0.5) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else None
    if threshold is not None:
        # Filtering records above the median
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"\nFiltered records where {numeric_field_id} > {threshold:.2f} (median): {len(filtered_df)} records\n")
        print(filtered_df.head(3))
        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized {numeric_field_id} (z-score) for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head(3))
        # Grouping
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nMean {numeric_field_id} by {group_field_id} in filtered data:")
            print(grouped_df.head())
else:
    print("No appropriate numeric field found for EDA.")

## 5. Visualization

Visualize distributions or relationships in the dataset using pandas and seaborn/matplotlib. For best interpretability, fields are referenced by their full `@id`s.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()
        plt.show()
else:
    print("Visualization skipped: No numeric field found.")

## 6. Conclusion

In this notebook, we demonstrated how to load, examine, and process a Croissant schema dataset using `mlcroissant`, referencing all dataset components by their canonical `@id` where possible. This ensures reproducibility and unambiguous referencing. You can further extend this notebook to perform model training, advanced statistics, or integrate additional Croissant datasets for richer data analysis.
